Experiment 1: Semantic Network with Inheritance

This notebook shows the knowledge base, inheritance queries with exceptions, the ostrich extension, a frame-based reimplementation, and a simple multiple-parent order.

In [7]:
relations = [
    ('penguin', 'is', 'bird'),
    ('ostrich', 'is', 'bird'),
    ('duck', 'is', 'swimmer'),
    ('duck', 'is', 'bird'),
    ('bird', 'is', 'animal'),
    ('shark', 'is', 'fish'),
    ('fish', 'is', 'animal'),
    ('swimmer', 'is', 'animal')
]

facts = [
    ('bird', 'can', 'fly'),
    ('animal', 'has', 'skin'),
    ('penguin', 'cannot', 'fly'),
    ('ostrich', 'cannot', 'fly'),
    ('ostrich', 'can', 'run fast'),
    ('swimmer', 'can', 'swim')
]


def parents_of(node):
    return [obj for subj, rel, obj in relations if subj == node and rel == 'is']


def inheritance_chain(node):
    chain = [node]
    current = node
    while True:
        parents = parents_of(current)
        if not parents:
            return chain
        current = parents[0]
        chain.append(current)


def local_facts(node):
    return [(rel, obj) for subj, rel, obj in facts if subj == node]


def query(node, relation, obj=None):
    print(f"Query: {node} {relation}" + (f" {obj}" if obj else ""))

    def search(current):
        entries = local_facts(current)
        if obj is not None:
            for rel, value in entries:
                if rel == relation and value == obj:
                    return f"{rel} {value}"
            if relation == 'can':
                for rel, value in entries:
                    if rel == 'cannot' and value == obj:
                        return f"cannot {value}"
        else:
            for rel, value in entries:
                if rel == relation:
                    return f"{rel} {value}"
            if relation == 'can':
                for rel, value in entries:
                    if rel == 'cannot':
                        return f"cannot {value}"
        for parent in parents_of(current):
            answer = search(parent)
            if answer != 'unknown':
                return answer
        return 'unknown'

    return search(node)

In [8]:
print('Knowledge base:')
print('Triples:')
for triple in relations:
    print(triple)

print('Facts:')
for fact in facts:
    print(fact)

print()
print('Inheritance chain for penguin:', ' -> '.join(inheritance_chain('penguin')))
print('penguin can fly:', query('penguin', 'can', 'fly'))
print('shark has skin:', query('shark', 'has not'))
print('ostrich can fly:', query('ostrich', 'can', 'fly'))
print('ostrich can run fast:', query('ostrich', 'can', 'run fast'))
print('duck can:', query('duck', 'can'))

Knowledge base:
Triples:
('penguin', 'is', 'bird')
('ostrich', 'is', 'bird')
('duck', 'is', 'swimmer')
('duck', 'is', 'bird')
('bird', 'is', 'animal')
('shark', 'is', 'fish')
('fish', 'is', 'animal')
('swimmer', 'is', 'animal')
Facts:
('bird', 'can', 'fly')
('animal', 'has', 'skin')
('penguin', 'cannot', 'fly')
('ostrich', 'cannot', 'fly')
('ostrich', 'can', 'run fast')
('swimmer', 'can', 'swim')

Inheritance chain for penguin: penguin -> bird -> animal
Query: penguin can fly
penguin can fly: cannot fly
Query: shark has not
shark has skin: unknown
Query: ostrich can fly
ostrich can fly: cannot fly
Query: ostrich can run fast
ostrich can run fast: can run fast
Query: duck can
duck can: can swim


In [9]:
frames = {
    'animal': {'slots': {'has': 'skin'}, 'ako': []},
    'fish': {'slots': {}, 'ako': ['animal']},
    'bird': {'slots': {'can': 'fly'}, 'ako': ['animal']},
    'swimmer': {'slots': {'can': 'swim'}, 'ako': ['animal']},
    'penguin': {'slots': {'cannot': 'fly'}, 'ako': ['bird']},
    'ostrich': {'slots': {'cannot': 'fly', 'can': 'run fast'}, 'ako': ['bird']},
    'shark': {'slots': {}, 'ako': ['fish']},
    'duck': {'slots': {}, 'ako': ['swimmer', 'bird']}
}


def frame_lookup(frame_name, relation, obj=None):
    frame = frames[frame_name]
    slots = frame['slots']
    if obj is not None:
        if relation in slots and slots[relation] == obj:
            return f"{relation} {obj}"
        if relation == 'can' and 'cannot' in slots and slots['cannot'] == obj:
            return f"cannot {obj}"
    else:
        if relation in slots:
            return f"{relation} {slots[relation]}"
        if relation == 'can' and 'cannot' in slots:
            return f"cannot {slots['cannot']}"
    for parent in frame['ako']:
        answer = frame_lookup(parent, relation, obj)
        if answer != 'unknown':
            return answer
    return 'unknown'


print('Frames:')
for name, frame in frames.items():
    print(name, frame)

print('Frame penguin can fly:', frame_lookup('penguin', 'can', 'fly'))
print('Frame ostrich can run fast:', frame_lookup('ostrich', 'can', 'run fast'))
print('Frame duck can:', frame_lookup('duck', 'can'))

Frames:
animal {'slots': {'has': 'skin'}, 'ako': []}
fish {'slots': {}, 'ako': ['animal']}
bird {'slots': {'can': 'fly'}, 'ako': ['animal']}
swimmer {'slots': {'can': 'swim'}, 'ako': ['animal']}
penguin {'slots': {'cannot': 'fly'}, 'ako': ['bird']}
ostrich {'slots': {'cannot': 'fly', 'can': 'run fast'}, 'ako': ['bird']}
shark {'slots': {}, 'ako': ['fish']}
duck {'slots': {}, 'ako': ['swimmer', 'bird']}
Frame penguin can fly: cannot fly
Frame ostrich can run fast: can run fast
Frame duck can: can swim


Observations

`penguin can fly` returns `cannot fly` because the local exception is checked before the inherited `bird can fly` fact.

The result is non-monotonic: adding a more specific fact changes what the system concludes.

For multiple parents, I search parents in the order they are listed so the lookup stays deterministic while still preferring local knowledge first.